# Reporte de Auditoría y Cambios Sugeridos · Módulo de API y Servidor

### 1. Archivo: src/storage.py
Nombre del error: Volatilidad de estado por almacenamiento en memoria (In-Memory State Loss).

Explicación de lo que tenemos: El script gestiona las transacciones pendientes de revisión y el historial de feedback utilizando diccionarios y listas nativas de Python (_pending_transactions, _feedback_history). Debido a que hemos desplegado la API dentro de un contenedor Docker en AWS EC2, cualquier reinicio del contenedor, fallo del proceso de Uvicorn o escalado de servidores destruirá por completo la cola de trabajo de los analistas humanos.

Explicación del cambio a realizar: Aunque para el despliegue inmediato mantengamos la deuda técnica para dar servicio a Full Stack, se debe documentar la migración hacia una base de datos relacional (como PostgreSQL en AWS RDS) o una base de datos embebida ligera (sqlite3) con un volumen persistente mapeado en Docker.

Líneas de código que se modifican porque están mal (Estructuras volátiles globales):

    # Almacenamiento volátil en RAM dentro de src/storage.py
    _pending_transactions = {}
    _feedback_history = []

    def save_pending_transaction(tx_id: str, tx_data: dict):
        _pending_transactions[tx_id] = tx_data


Líneas de código que estarían mejor (Evolución persistente recomendada):

    import sqlite3

    DB_PATH = "data/sentinel_storage.db"

    def save_pending_transaction(tx_id: str, tx_data: dict):
        # Persistencia real en disco atómica para evitar pérdidas en AWS
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(
            "INSERT OR REPLACE INTO pending_transactions (id, data) VALUES (?, ?)",
            (tx_id, str(tx_data))
        )
        conn.commit()
        conn.close()